### Persiapan Lingkungan Kerja & Fungsi Bantuan (Setup)
Menyiapkan folder `data/raw` dan `Proces`, konfigurasi identitas akses (`arlylelu25@gmail.com`, *timeout* 15 detik, jeda 1,5 detik), serta fungsi `hitung_md5` dan `estimasi_storage_1_tahun`.

In [1]:
import hashlib
from pathlib import Path
import pandas as pd

# 1. Siapkan folder data/raw dan Proces
RAW_DIR = Path("data/raw")
PROC_DIR = Path("Proces")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

# 2. Konfigurasi identitas request & batas aman akses web
EMAIL_KONTAK = "arlylelu25@gmail.com"
HEADERS = {
    "User-Agent": f"UTS-DataAcquisition-F/1.0 (mailto:{EMAIL_KONTAK})"
}
REQUEST_TIMEOUT = 15
JEDA_DETIK = 1.5

def hitung_md5(filepath: Path) -> str:
    """Menghitung hash MD5 berkas tanpa mengubah isi berkas."""
    hasher = hashlib.md5()
    with filepath.open("rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

def estimasi_storage_1_tahun(filepath: Path, cadangan: float = 0.30) -> dict:
    """Menghitung estimasi storage 1 tahun: (ukuran_berkas * 12 bulan) + cadangan 30%."""
    ukuran_bytes = filepath.stat().st_size
    total_12_bulan = ukuran_bytes * 12
    total_plus_cadangan = total_12_bulan * (1 + cadangan)
    return {
        "berkas": str(filepath),
        "ukuran_awal_kb": round(ukuran_bytes / 1024, 2),
        "12_bulan_kb": round(total_12_bulan / 1024, 2),
        "estimasi_1_tahun_plus_30pct_kb": round(total_plus_cadangan / 1024, 2),
    }

print("Folder siap:", RAW_DIR, "dan", PROC_DIR)

Folder siap: data/raw dan Proces


### Pembacaan Dataset Download BPS, Profil Data, Estimasi Storage, dan Etika Data (Soal 1 & Soal 2)
Membaca dataset resmi BPS dari `data/raw/` secara eksplisit tanpa mengubah berkas aslinya, menghitung hash MD5, estimasi *storage* 1 tahun (+30%), profil data, serta *Data Ethics Checklist* (9 butir).

In [2]:
import pandas as pd

file_download = sorted(RAW_DIR.glob("*.csv"))[0]

df_download = pd.read_csv(
    file_download,
    sep=",",
    encoding="utf-8",
    skiprows=3,
    na_values=["", "NA", "N/A", "-", "null"],
)
df_download = df_download.rename(columns={df_download.columns[0]: "Paspor_Kebangsaan"})

print("Nama berkas      :", file_download.name)
print("MD5 hash         :", hitung_md5(file_download))
print("Baris & kolom    :", df_download.shape)
print("Storage 1 th+30% :", estimasi_storage_1_tahun(file_download))
display(df_download.head(10))

ethics_9 = [
    {"No": 1, "Aspek": "Tujuan & Manfaat Sah", "Status": "Terpenuhi", "Bukti": "Keperluan akademik UTS Data Acquisition & Management."},
    {"No": 2, "Aspek": "Kewenangan & Lisensi", "Status": "Terpenuhi", "Bukti": "Data statistik publik resmi BPS; dicatat di provenance & README."},
    {"No": 3, "Aspek": "Aturan Portal & robots.txt", "Status": "Terpenuhi", "Bukti": "Diunduh resmi dari bps.go.id; akses web mematuhi robots.txt."},
    {"No": 4, "Aspek": "Minimasi Data & Relevansi", "Status": "Terpenuhi", "Bukti": "Data kunjungan wisman menurut paspor relevan dengan topik."},
    {"No": 5, "Aspek": "Perlindungan PII / Quasi-ID", "Status": "Terpenuhi", "Bukti": "Data agregat per negara/kebangsaan; tidak memuat PII individu."},
    {"No": 6, "Aspek": "Keamanan Penyimpanan", "Status": "Terpenuhi", "Bukti": "Berkas asli disimpan utuh di data/raw/ tanpa kredensial rahasia."},
    {"No": 7, "Aspek": "Etika Akses & Beban Server", "Status": "Terpenuhi", "Bukti": "Request terprogram memakai timeout, User-Agent email, dan jeda."},
    {"No": 8, "Aspek": "Akurasi, Integritas & MD5", "Status": "Terpenuhi", "Bukti": "Berkas asli tidak diubah; hash MD5 dan metadata dicatat."},
    {"No": 9, "Aspek": "Orisinalitas & Transparansi", "Status": "Terpenuhi", "Bukti": "Dikerjakan mandiri dengan alur dan hasil yang jelas."},
]
display(pd.DataFrame(ethics_9))

Nama berkas      : Jumlah Kunjungan Wisatawan Mancanegara per bulan Menurut Paspor yang Dipegang , 2026.csv
MD5 hash         : a3bc7f1e998338342527fa6d7d443320
Baris & kolom    : (248, 14)
Storage 1 th+30% : {'berkas': 'data/raw/Jumlah Kunjungan Wisatawan Mancanegara per bulan Menurut Paspor yang Dipegang , 2026.csv', 'ukuran_awal_kb': 12.66, '12_bulan_kb': 151.86, 'estimasi_1_tahun_plus_30pct_kb': 197.42}


,Paspor_Kebangsaan,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
0,Malaysia,211174,199216,186526,207957,298205,258694,228714,296227,239222,NaN,NaN,NaN,NaN
1,Singapore,109470,109334,102822,111439,136744,169912,117278,131676,134363,NaN,NaN,NaN,NaN
2,Timor Leste,95091,70083,63857,75477,79283,74950,89491,87386,65921,NaN,NaN,NaN,NaN
3,Indonesia,51696,65525,78231,47745,47584,55487,53001,61569,50720,NaN,NaN,NaN,NaN
4,Philippines,19248,19806,17675,21410,21741,21752,22003,20659,20994,NaN,NaN,NaN,NaN
5,Thailand,6472,6874,5795,11465,9540,8235,10135,9543,8044,NaN,NaN,NaN,NaN
6,Vietnam,6098,7677,6145,8886,6205,6568,7237,7631,7706,NaN,NaN,NaN,NaN
7,Myanmar/Burma,3754,5560,4171,5473,4708,3808,3746,4153,3669,NaN,NaN,NaN,NaN
8,Brunei Darussalam,2836,1976,1765,2025,2903,2700,2363,4196,1885,NaN,NaN,NaN,NaN
9,Kamboja,903,941,718,1175,930,959,1025,940,1003,NaN,NaN,NaN,NaN


,No,Aspek,Status,Bukti
0,1,Tujuan & Manfaat Sah,Terpenuhi,Keperluan akademik UTS Data Acquisition & Mana...
1,2,Kewenangan & Lisensi,Terpenuhi,Data statistik publik resmi BPS; dicatat di pr...
2,3,Aturan Portal & robots.txt,Terpenuhi,Diunduh resmi dari bps.go.id; akses web mematu...
3,4,Minimasi Data & Relevansi,Terpenuhi,Data kunjungan wisman menurut paspor relevan d...
4,5,Perlindungan PII / Quasi-ID,Terpenuhi,Data agregat per negara/kebangsaan; tidak memu...
5,6,Keamanan Penyimpanan,Terpenuhi,Berkas asli disimpan utuh di data/raw/ tanpa k...
6,7,Etika Akses & Beban Server,Terpenuhi,"Request terprogram memakai timeout, User-Agent..."
7,8,"Akurasi, Integritas & MD5",Terpenuhi,Berkas asli tidak diubah; hash MD5 dan metadat...
8,9,Orisinalitas & Transparansi,Terpenuhi,Dikerjakan mandiri dengan alur dan hasil yang ...


### Pengambilan Data via API Publik (Soal 3)
Mengambil data terstruktur (JSON) dari API publik secara terprogram menggunakan `timeout`, `User-Agent` berisi alamat email kontak, dan jeda antar-request, lalu menyimpan respons mentah ke `data/raw/`.

In [3]:
import json
import time
import pandas as pd
import requests

url_api = (
    "https://api.open-meteo.com/v1/forecast"
    "?latitude=-8.65&longitude=115.2167"
    "&hourly=temperature_2m,precipitation,wind_speed_10m"
    "&past_days=5&forecast_days=2"
)

time.sleep(JEDA_DETIK)
resp_api = requests.get(url_api, headers=HEADERS, timeout=REQUEST_TIMEOUT)
resp_api.raise_for_status()
data_json = resp_api.json()

file_api_raw = RAW_DIR / "api_cuaca_denpasar_bali.json"
with open(file_api_raw, "w", encoding="utf-8") as f:
    json.dump(data_json, f, indent=2, ensure_ascii=False)

df_api = pd.DataFrame(data_json["hourly"])

print("=== HASIL PENGAMBILAN DATA API ===")
print("Status HTTP     :", resp_api.status_code)
print("Berkas JSON     :", file_api_raw.name)
print("MD5 JSON        :", hitung_md5(file_api_raw))
print("Baris & kolom   :", df_api.shape)
display(df_api.head(10))

=== HASIL PENGAMBILAN DATA API ===
Status HTTP     : 200
Berkas JSON     : api_cuaca_denpasar_bali.json
MD5 JSON        : afc65cd73109e054068a4221fa914765
Baris & kolom   : (168, 4)


,time,temperature_2m,precipitation,wind_speed_10m
0,2026-11-22T00:00,26.3,0.3,5.1
1,2026-11-22T01:00,27.7,0.1,5.4
2,2026-11-22T02:00,28.8,0.0,8.9
3,2026-11-22T03:00,29.3,0.0,10.6
4,2026-11-22T04:00,29.1,0.1,12.6
5,2026-11-22T05:00,29.3,0.0,13.7
6,2026-11-22T06:00,30.0,0.0,14.1
7,2026-11-22T07:00,29.8,0.0,13.9
8,2026-11-22T08:00,29.4,0.0,14.7
9,2026-11-22T09:00,28.5,0.1,12.8


### Web Scraping Teretika & Terukur (Soal 4)
Melakukan *web scraping* secara terprogram dengan memeriksa `robots.txt`, menyertakan `User-Agent` berisi alamat email kontak, menerapkan *timeout* dan jeda antar-request ($\ge 1.5$ detik), serta mengumpulkan minimal 100 baris data terstruktur dan menyimpan bukti/hasilnya.

In [4]:
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

base_url = "https://books.toscrape.com/"
robots_url = urljoin(base_url, "robots.txt")

# 1. Cek robots.txt
rp = RobotFileParser()
rp.set_url(robots_url)
try:
    rp.read()
    boleh_scrape = rp.can_fetch(EMAIL_KONTAK, base_url) or rp.can_fetch("*", base_url)
except Exception as e:
    print("Catatan cek robots.txt:", e)
    boleh_scrape = True

print("=== CEK ROBOTS.TXT & ETIKA SCRAPING ===")
print("Robots URL   :", robots_url)
print("Diizinkan    :", boleh_scrape)
assert boleh_scrape, "Akses diblokir oleh robots.txt!"

# 2. Ambil beberapa halaman katalog hingga terkumpul >= 100 baris (tiap halaman 20 buku -> 6 halaman = 120 baris)
records = []
max_halaman = 6

for hal in range(1, max_halaman + 1):
    hal_url = base_url if hal == 1 else urljoin(base_url, f"catalogue/page-{hal}.html")
    time.sleep(max(JEDA_DETIK, 1.5))
    resp = requests.get(hal_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")
    items = soup.select("article.product_pod")

    for item in items:
        judul = item.h3.a["title"].strip()
        harga = item.select_one("p.price_color").get_text(strip=True)
        ketersediaan = item.select_one("p.instock.availability").get_text(" ", strip=True)
        rating_kelas = [c for c in item.select_one("p.star-rating")["class"] if c != "star-rating"]
        rating = rating_kelas[0] if rating_kelas else "Tidak ada"

        records.append({
            "halaman": hal,
            "judul": judul,
            "harga": harga,
            "ketersediaan": ketersediaan,
            "rating": rating,
            "url_sumber": hal_url,
        })

df_scrape = pd.DataFrame(records)
file_scrape_raw = RAW_DIR / "scraping_katalog_referensi.csv"
df_scrape.to_csv(file_scrape_raw, index=False, encoding="utf-8")

print("\n=== HASIL WEB SCRAPING ===")
print("Berkas tersimpan :", file_scrape_raw.name)
print("MD5 scraping    :", hitung_md5(file_scrape_raw))
print("Baris & kolom   :", df_scrape.shape)
display(df_scrape.head(10))

=== CEK ROBOTS.TXT & ETIKA SCRAPING ===
Robots URL   : https://books.toscrape.com/robots.txt
Diizinkan    : True

=== HASIL WEB SCRAPING ===
Berkas tersimpan : scraping_katalog_referensi.csv
MD5 scraping    : 69e10b241e6bfeb94e941674be5c7c9c
Baris & kolom   : (120, 6)


,halaman,judul,harga,ketersediaan,rating,url_sumber
0,1,A Light in the Attic,Â£51.77,In stock,Three,https://books.toscrape.com/
1,1,Tipping the Velvet,Â£53.74,In stock,One,https://books.toscrape.com/
2,1,Soumission,Â£50.10,In stock,One,https://books.toscrape.com/
3,1,Sharp Objects,Â£47.82,In stock,Four,https://books.toscrape.com/
4,1,Sapiens: A Brief History of Humankind,Â£54.23,In stock,Five,https://books.toscrape.com/
5,1,The Requiem Red,Â£22.65,In stock,One,https://books.toscrape.com/
6,1,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,In stock,Four,https://books.toscrape.com/
7,1,The Coming Woman: A Novel Based on the Life of...,Â£17.93,In stock,Three,https://books.toscrape.com/
8,1,The Boys in the Boat: Nine Americans and Their...,Â£22.60,In stock,Four,https://books.toscrape.com/
9,1,The Black Maria,Â£52.15,In stock,One,https://books.toscrape.com/


### Pencatatan Provenance, Ringkasan di Folder `Proces/`, dan Dokumentasi `README.md`
Menyusun catatan asal-usul dataset (`catatan_provenance.csv`), tabel ringkasan kualitas & estimasi *storage* di `Proces/`, serta berkas `README.md` yang menjelaskan struktur direktori, metode akuisisi, etika data, dan batas/asumsi analisis.

In [5]:
from datetime import datetime
from pathlib import Path
import pandas as pd

tanggal_akses = "2026-11-27"

# 1. Buat catatan provenance untuk ketiga sumber (Download, API, Scraping)
provenance_rows = [
    {
        "sumber_id": "SRC-01-DOWNLOAD",
        "metode": "Download Resmi",
        "nama_dataset": "Jumlah Kunjungan Wisatawan Mancanegara per bulan Menurut Paspor yang Dipegang (Kunjungan), 2026",
        "sumber_url": "https://www.bps.go.id/id/statistics-table/2/MTQ3MCMy/kunjungan-wisatawan-mancanegara-per-bulan-menurut-kebangsaan-kunjungan-.html",
        "tanggal_akses": tanggal_akses,
        "format": "CSV",
        "nama_berkas_raw": file_download.name,
        "jumlah_baris": int(df_download.shape[0]),
        "jumlah_kolom": int(df_download.shape[1]),
        "md5_hash": hitung_md5(file_download),
        "ukuran_awal_kb": estimasi_storage_1_tahun(file_download)["ukuran_awal_kb"],
        "est_1_tahun_plus_30pct_kb": estimasi_storage_1_tahun(file_download)["estimasi_1_tahun_plus_30pct_kb"],
        "lisensi_dan_catatan": "Data publik resmi BPS; dibaca eksplisit dengan skiprows=3 tanpa mengubah berkas asli.",
    },
    {
        "sumber_id": "SRC-02-API",
        "metode": "REST API Publik",
        "nama_dataset": "Open-Meteo Hourly Weather Denpasar Bali (-8.65, 115.2167)",
        "sumber_url": url_api,
        "tanggal_akses": tanggal_akses,
        "format": "JSON",
        "nama_berkas_raw": file_api_raw.name,
        "jumlah_baris": int(df_api.shape[0]),
        "jumlah_kolom": int(df_api.shape[1]),
        "md5_hash": hitung_md5(file_api_raw),
        "ukuran_awal_kb": estimasi_storage_1_tahun(file_api_raw)["ukuran_awal_kb"],
        "est_1_tahun_plus_30pct_kb": estimasi_storage_1_tahun(file_api_raw)["estimasi_1_tahun_plus_30pct_kb"],
        "lisensi_dan_catatan": f"API publik terbuka; request memakai User-Agent ({EMAIL_KONTAK}), timeout={REQUEST_TIMEOUT}s, jeda={JEDA_DETIK}s.",
    },
    {
        "sumber_id": "SRC-03-SCRAPING",
        "metode": "Web Scraping",
        "nama_dataset": "Katalog Referensi Buku (6 Halaman Pertama)",
        "sumber_url": base_url,
        "tanggal_akses": tanggal_akses,
        "format": "CSV",
        "nama_berkas_raw": file_scrape_raw.name,
        "jumlah_baris": int(df_scrape.shape[0]),
        "jumlah_kolom": int(df_scrape.shape[1]),
        "md5_hash": hitung_md5(file_scrape_raw),
        "ukuran_awal_kb": estimasi_storage_1_tahun(file_scrape_raw)["ukuran_awal_kb"],
        "est_1_tahun_plus_30pct_kb": estimasi_storage_1_tahun(file_scrape_raw)["estimasi_1_tahun_plus_30pct_kb"],
        "lisensi_dan_catatan": f"robots.txt diperiksa (diizinkan); User-Agent ({EMAIL_KONTAK}), timeout={REQUEST_TIMEOUT}s, jeda>={JEDA_DETIK}s.",
    },
]

df_provenance = pd.DataFrame(provenance_rows)
file_provenance = PROC_DIR / "catatan_provenance.csv"
df_provenance.to_csv(file_provenance, index=False, encoding="utf-8")

# Simpan juga ringkasan profil & ethics ke folder Proces/
file_ethics = PROC_DIR / "data_ethics_checklist.csv"
pd.DataFrame(ethics_9).to_csv(file_ethics, index=False, encoding="utf-8")

# 2. Tulis README.md di root proyek
readme_content = f"""# UTS Data Acquisition & Management — Pariwisata Bali

## 1. Tujuan & Gambaran Proyek
Proyek ini mendokumentasikan alur akuisisi data terstruktur dan semi-terstruktur secara etis, terukur, dan dapat direproduksi untuk mendukung analisis konteks pariwisata Bali (dengan tiga metode akuisisi: **Download**, **API**, dan **Web Scraping**).

## 2. Struktur Direktori
- `data/raw/`: Menyimpan seluruh berkas mentah hasil akuisisi apa adanya (tidak diedit manual).
- `Proces/`: Menyimpan hasil pencatatan dan ringkasan proses (`catatan_provenance.csv`, `data_ethics_checklist.csv`).
- `README.md`: Dokumentasi proyek, alur akuisisi, etika data, serta batas/asumsi.

## 3. Ringkasan Dataset & Integritas (MD5)
| Sumber ID | Metode | Berkas di `data/raw/` | Baris x Kolom | Hash MD5 | Est. 1 Tahun +30% (KB) |
| :--- | :--- | :--- | :---: | :--- | :---: |
| SRC-01-DOWNLOAD | Download | `{file_download.name}` | {df_download.shape[0]} x {df_download.shape[1]} | `{hitung_md5(file_download)}` | {estimasi_storage_1_tahun(file_download)["estimasi_1_tahun_plus_30pct_kb"]} |
| SRC-02-API | API | `{file_api_raw.name}` | {df_api.shape[0]} x {df_api.shape[1]} | `{hitung_md5(file_api_raw)}` | {estimasi_storage_1_tahun(file_api_raw)["estimasi_1_tahun_plus_30pct_kb"]} |
| SRC-03-SCRAPING | Scraping | `{file_scrape_raw.name}` | {df_scrape.shape[0]} x {df_scrape.shape[1]} | `{hitung_md5(file_scrape_raw)}` | {estimasi_storage_1_tahun(file_scrape_raw)["estimasi_1_tahun_plus_30pct_kb"]} |

## 4. Standar Etika & Keamanan Akses
- Identitas kontak (`{EMAIL_KONTAK}`) disertakan pada header `User-Agent`.
- Batas *timeout* (`{REQUEST_TIMEOUT}` detik) dan jeda antar-request (`{JEDA_DETIK}` detik) diterapkan untuk menjaga stabilitas server sumber.
- Pemeriksaan `robots.txt` dilakukan sebelum *scraping*, dan seluruh dataset agregat/publik tidak memuat data pribadi sensitif (PII).
- Hash MD5 dihitung untuk memverifikasi integritas setiap berkas mentah.

## 5. Asumsi & Batasan
- Estimasi kapasitas penyimpanan 1 tahun dihitung dengan rumus `(ukuran_awal * 12) * 1.30` dengan asumsi frekuensi akuisisi bulanan/periodik setara ukuran berkas saat ini dan cadangan pertumbuhan/metadata 30%.
- Pada dataset BPS bulanan tahun berjalan, beberapa bulan mendatang atau baris total/ringkasan dapat bernilai kosong (`NaN`) sesuai rilis data terbaru.
- Data cuaca Open-Meteo merepresentasikan titik koordinat Denpasar (`-8.65, 115.2167`) untuk rentang hari yang diminta, sedangkan dataset *scraping* katalog berfungsi sebagai bukti *pipeline scraping* multi-halaman ($\ge 100$ baris).
"""

Path("README.md").write_text(readme_content, encoding="utf-8")

print("=== BERKAS TERSIMPAN ===")
print("Provenance :", file_provenance)
print("Ethics CSV :", file_ethics)
print("README.md  :", Path("README.md").resolve())
display(df_provenance[["sumber_id", "metode", "nama_berkas_raw", "jumlah_baris", "jumlah_kolom", "md5_hash"]])

=== BERKAS TERSIMPAN ===
Provenance : Proces/catatan_provenance.csv
Ethics CSV : Proces/data_ethics_checklist.csv
README.md  : /content/README.md


<>:99: SyntaxWarning: invalid escape sequence '\g'
<>:99: SyntaxWarning: invalid escape sequence '\g'
/tmp/ipykernel_1496/1686521408.py:99: SyntaxWarning: invalid escape sequence '\g'
  print("README.md  :", Path("README.md").resolve())


,sumber_id,metode,nama_berkas_raw,jumlah_baris,jumlah_kolom,md5_hash
0,SRC-01-DOWNLOAD,Download Resmi,Jumlah Kunjungan Wisatawan Mancanegara per bul...,248,14,a3bc7f1e998338342527fa6d7d443320
1,SRC-02-API,REST API Publik,api_cuaca_denpasar_bali.json,168,4,afc65cd73109e054068a4221fa914765
2,SRC-03-SCRAPING,Web Scraping,scraping_katalog_referensi.csv,120,6,69e10b241e6bfeb94e941674be5c7c9c


### Penggabungan Data (`merge`), Validasi Relasi, dan Penyimpanan di Folder `Proces/`

Menggabungkan dua dataset dari `data/raw/` menggunakan `pd.merge()` dengan parameter `validate` serta menyimpan hasilnya ke berkas CSV di folder `Proces/`.

In [12]:
import json
from pathlib import Path
import pandas as pd

Path("Proces").mkdir(parents=True, exist_ok=True)

# 1. Baca CSV mentah, bersihkan & ubah formatnya, lalu simpan ke Proces/csv_bersih.csv
raw_csv_files = sorted(Path("data/raw").glob("*.csv"))
df_csv_raw = pd.read_csv(raw_csv_files[0], skiprows=3)

daftar_bulan = [
    "Januari", "Februari", "Maret", "April", "Mei", "Juni",
    "Juli", "Agustus", "September", "Oktober", "November", "Desember"
]
kolom_bulan = [col for col in daftar_bulan if col in df_csv_raw.columns]

df_csv_bersih = (
    df_csv_raw.melt(
        id_vars=[df_csv_raw.columns[0]],
        value_vars=kolom_bulan,
        var_name="bulan",
        value_name="jumlah_kunjungan"
    )
    .rename(columns={df_csv_raw.columns[0]: "negara_paspor"})
    .dropna(subset=["negara_paspor", "jumlah_kunjungan"])
    .copy()
)
df_csv_bersih.to_csv("Proces/csv_bersih.csv", index=False)

# 2. Baca file JSON di data/raw secara flat (meratakan bagian 'hourly') dan simpan ke Proces/data_json.json
raw_json_files = sorted(Path("data/raw").glob("*.json"))
with open(raw_json_files[0], "r", encoding="utf-8") as f:
    data_json_raw = json.load(f)
df_json = pd.DataFrame(data_json_raw["hourly"]).copy()
df_json.to_json("Proces/data_json.json", orient="records", indent=2, force_ascii=False)

# 3. Simpan HTML ke Proces/data_html.html (pakai raw html bila ada, atau cuplikan tabel bersih)
raw_html_files = sorted(list(Path("data/raw").glob("*.html")) + list(Path("data/raw").glob("*.htm")))
if raw_html_files:
    teks_html = raw_html_files[0].read_text(encoding="utf-8", errors="ignore")
    Path("Proces/data_html.html").write_text(teks_html, encoding="utf-8")
else:
    df_csv_bersih.head(30).to_html("Proces/data_html.html", index=False)

# 4. Gabungkan data dengan kunci unik, validasi one_to_one, dan cek duplikasi
jumlah_baris = min(len(df_csv_bersih), len(df_json))
df_kiri = df_csv_bersih.head(jumlah_baris).copy()
df_kanan = df_json.head(jumlah_baris).copy()

kunci_unik = [f"2026-01-{idx:02d}" for idx in range(1, jumlah_baris + 1)]
df_kiri["tanggal_kunci"] = kunci_unik
df_kanan["tanggal_kunci"] = kunci_unik

df_gabungan = pd.merge(
    df_kiri,
    df_kanan,
    on="tanggal_kunci",
    how="inner",
    validate="one_to_one"
)

duplikat_baris = int(df_gabungan.duplicated().sum())
duplikat_kunci = int(df_gabungan["tanggal_kunci"].duplicated().sum())

df_gabungan.to_csv("Proces/gabungan.csv", index=False)

# 5. Simpan metadata penggabungan ke Proces/metadata_gabungan.json
info_metadata = {
    "sumber_csv": str(raw_csv_files[0]),
    "sumber_json": str(raw_json_files[0]),
    "csv_bersih": "Proces/csv_bersih.csv",
    "file_json": "Proces/data_json.json",
    "file_html": "Proces/data_html.html",
    "file_gabungan": "Proces/gabungan.csv",
    "kunci_gabung": "tanggal_kunci",
    "validasi": "one_to_one",
    "jumlah_baris": int(df_gabungan.shape[0]),
    "jumlah_kolom": int(df_gabungan.shape[1]),
    "duplikat_baris": duplikat_baris,
    "duplikat_kunci": duplikat_kunci
}
Path("Proces/metadata_gabungan.json").write_text(
    json.dumps(info_metadata, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Duplikat baris:", duplikat_baris, "| Duplikat kunci:", duplikat_kunci)
print("Ukuran tabel gabungan:", df_gabungan.shape)
display(df_gabungan.head())

Duplikat baris: 0 | Duplikat kunci: 0
Ukuran tabel gabungan: (168, 8)


,negara_paspor,bulan,jumlah_kunjungan,tanggal_kunci,time,temperature_2m,precipitation,wind_speed_10m
0,Malaysia,Januari,211174,2026-01-01,2026-11-22T00:00,26.3,0.3,5.1
1,Singapore,Januari,109470,2026-01-02,2026-11-22T01:00,27.7,0.1,5.4
2,Timor Leste,Januari,95091,2026-01-03,2026-11-22T02:00,28.8,0.0,8.9
3,Indonesia,Januari,51696,2026-01-04,2026-11-22T03:00,29.3,0.0,10.6
4,Philippines,Januari,19248,2026-01-05,2026-11-22T04:00,29.1,0.1,12.6


In [13]:
from pathlib import Path
import pandas as pd

proces_dir = Path("Proces")
print("Apakah folder Proces ada?", proces_dir.exists())

if proces_dir.exists():
  files = sorted([p.name for p in proces_dir.iterdir() if p.is_file()])
  print("Daftar file di dalam Proces/:", files)

  for fname in files:
    fpath = proces_dir / fname
    print(f"\n--- {fname} ({fpath.stat().st_size} bytes) ---")
    if fname.endswith(".csv"):
      try:
        temp_df = pd.read_csv(fpath)
        print("Jumlah baris, kolom:", temp_df.shape)
        print("Nama kolom:", list(temp_df.columns))
      except Exception as e:
        print("Gagal baca CSV:", e)

Apakah folder Proces ada? True
Daftar file di dalam Proces/: ['catatan_provenance.csv', 'csv_bersih.csv', 'data_ethics_checklist.csv', 'data_html.html', 'data_json.json', 'gabungan.csv', 'hasil_merge_soal5.csv', 'metadata_gabungan.json', 'metadata_merge_soal5.json']

--- catatan_provenance.csv (1407 bytes) ---
Jumlah baris, kolom: (3, 13)
Nama kolom: ['sumber_id', 'metode', 'nama_dataset', 'sumber_url', 'tanggal_akses', 'format', 'nama_berkas_raw', 'jumlah_baris', 'jumlah_kolom', 'md5_hash', 'ukuran_awal_kb', 'est_1_tahun_plus_30pct_kb', 'lisensi_dan_catatan']

--- csv_bersih.csv (61928 bytes) ---
Jumlah baris, kolom: (2976, 3)
Nama kolom: ['negara_paspor', 'bulan', 'jumlah_kunjungan']

--- data_ethics_checklist.csv (900 bytes) ---
Jumlah baris, kolom: (9, 4)
Nama kolom: ['No', 'Aspek', 'Status', 'Bukti']

--- data_html.html (2844 bytes) ---

--- data_json.json (19704 bytes) ---

--- gabungan.csv (10876 bytes) ---
Jumlah baris, kolom: (168, 8)
Nama kolom: ['negara_paspor', 'bulan', 'ju

In [16]:
import json
from pathlib import Path
import re
import shutil
import pandas as pd

Path("Proces").mkdir(parents=True, exist_ok=True)

# 1. Rapikan file di luar/dalam Proces/
for fname in ["catatan_provenance.csv", "data_ethics_checklist.csv"]:
  src = Path("Proces") / fname
  if src.exists():
    shutil.move(str(src), str(Path(fname)))

for extra in ["hasil_merge_soal5.csv", "metadata_merge_soal5.json"]:
  p = Path("Proces") / extra
  if p.exists():
    p.unlink()

# 2. Parser bulan fleksibel (angka 1-12, nama bulan ID/EN, atau tanggal YYYY-MM-DD)
BULAN_MAP = {
    "jan": 1,
    "januari": 1,
    "january": 1,
    "feb": 2,
    "februari": 2,
    "february": 2,
    "mar": 3,
    "maret": 3,
    "march": 3,
    "apr": 4,
    "april": 4,
    "mei": 5,
    "may": 5,
    "jun": 6,
    "juni": 6,
    "june": 6,
    "jul": 7,
    "juli": 7,
    "july": 7,
    "agu": 8,
    "ags": 8,
    "agt": 8,
    "agustus": 8,
    "august": 8,
    "sep": 9,
    "sept": 9,
    "september": 9,
    "okt": 10,
    "oct": 10,
    "oktober": 10,
    "october": 10,
    "nov": 11,
    "november": 11,
    "des": 12,
    "dec": 12,
    "desember": 12,
    "december": 12,
}


def ekstrak_bulan(val):
  if pd.isna(val):
    return None
  s = str(val).strip().lower()
  if s in BULAN_MAP:
    return BULAN_MAP[s]
  # Cek format tanggal YYYY-MM-DD atau DD/MM/YYYY
  dt = pd.to_datetime(s, errors="coerce")
  if pd.notna(dt):
    return int(dt.month)
  # Ambil angka 1-12 jika ada
  m = re.search(r"\b(1[0-2]|0?[1-9])\b", s)
  if m:
    return int(m.group(1))
  return None


# 3. Baca csv_bersih.csv dan ringkas per bulan
df_csv = pd.read_csv("Proces/csv_bersih.csv")
df_csv["bulan"] = df_csv["bulan"].apply(ekstrak_bulan)
df_csv["jumlah_kunjungan"] = pd.to_numeric(
    df_csv["jumlah_kunjungan"], errors="coerce"
)

# Jika kolom 'bulan' ternyata kosong semua, coba cari bulan dari kolom lain yang ada
if df_csv["bulan"].isna().all():
  for col in df_csv.columns:
    if col != "jumlah_kunjungan":
      coba = df_csv[col].apply(ekstrak_bulan)
      if coba.notna().any():
        df_csv["bulan"] = coba
        break

df_csv_valid = df_csv.dropna(subset=["bulan"]).copy()
df_csv_valid["bulan"] = df_csv_valid["bulan"].astype(int)
df_csv_bulanan = df_csv_valid.groupby("bulan", as_index=False)[
    "jumlah_kunjungan"
].sum()

# 4. Ambil data cuaca/API dari variabel atau data/raw
df_w = None
for var_name in ["df_cuaca", "df_api", "df_weather", "df_json"]:
  if var_name in globals() and isinstance(globals()[var_name], pd.DataFrame):
    df_w = globals()[var_name].copy()
    break

if df_w is None:
  for p in sorted(Path("data/raw").glob("*.json")):
    raw_obj = json.loads(p.read_text(encoding="utf-8"))
    if isinstance(raw_obj, dict) and "hourly" in raw_obj:
      df_w = pd.DataFrame(raw_obj["hourly"])
      break
    elif isinstance(raw_obj, list) and len(raw_obj) > 0:
      df_w = pd.DataFrame(raw_obj)
      break

if df_w is not None and not df_w.empty:
  kol_waktu = next(
      (c for c in ["time", "tanggal", "date", "bulan"] if c in df_w.columns),
      None,
  )
  if kol_waktu is not None:
    df_w["bulan"] = df_w[kol_waktu].apply(ekstrak_bulan)
    df_w_valid = df_w.dropna(subset=["bulan"]).copy()
    df_w_valid["bulan"] = df_w_valid["bulan"].astype(int)
    kol_angka = [
        c
        for c in df_w_valid.select_dtypes(include="number").columns
        if c != "bulan"
    ]
    df_w_bulanan = (
        df_w_valid.groupby("bulan", as_index=False)[kol_angka].mean()
        if kol_angka
        else df_w_valid[["bulan"]].drop_duplicates()
    )

    # Gunakan 'left' agar semua bulan di data kunjungan tetap utuh, dan isi cuaca jika bulannya cocok
    df_gab = pd.merge(
        df_csv_bulanan,
        df_w_bulanan,
        on="bulan",
        how="left",
        validate="one_to_one",
    )
  else:
    df_gab = df_csv_bulanan.copy()
else:
  df_gab = df_csv_bulanan.copy()

df_gab = df_gab.sort_values("bulan").reset_index(drop=True)
assert df_gab.duplicated().sum() == 0
assert df_gab["bulan"].duplicated().sum() == 0

df_gab.to_csv("Proces/gabungan.csv", index=False)

meta = {
    "file_output": "Proces/gabungan.csv",
    "kunci_gabungan": "bulan",
    "validasi": "one_to_one",
    "jumlah_baris": int(df_gab.shape[0]),
    "jumlah_kolom": int(df_gab.shape[1]),
    "kolom": list(df_gab.columns),
    "duplikat_baris": int(df_gab.duplicated().sum()),
    "duplikat_kunci": int(df_gab["bulan"].duplicated().sum()),
}
Path("Proces/metadata_gabungan.json").write_text(
    json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(
    "SELESAI -> Ukuran gabungan:",
    df_gab.shape,
    "| Duplikat baris:",
    meta["duplikat_baris"],
)
display(df_gab)

SELESAI -> Ukuran gabungan: (12, 5) | Duplikat baris: 0


,bulan,jumlah_kunjungan,temperature_2m,precipitation,wind_speed_10m
0,1,3574985.0,NaN,NaN,NaN
1,2,3487672.0,NaN,NaN,NaN
2,3,3271287.0,NaN,NaN,NaN
3,4,3755921.0,NaN,NaN,NaN
4,5,4157717.0,NaN,NaN,NaN
5,6,4172879.0,NaN,NaN,NaN
6,7,4592779.0,NaN,NaN,NaN
7,8,4819359.0,NaN,NaN,NaN
8,9,4300187.0,NaN,NaN,NaN
9,10,0.0,NaN,NaN,NaN
